# Data Cleaning and Missing Values with Pandas

This notebook introduces a systematic data-cleaning workflow using a small, intentionally imperfect retail dataset.

## Learning objectives

After completing this notebook, students should be able to:

- inspect the structure and data types of a dataset;
- identify missing, duplicated, inconsistent, and invalid values;
- convert columns to suitable data types;
- clean inconsistent categorical labels;
- choose an appropriate missing-value treatment;
- detect suspicious numerical values and simple outliers;
- validate the cleaned dataset;
- document cleaning decisions and their limitations.

## Required libraries

- Python 3
- Pandas
- NumPy
- Matplotlib

The dataset is generated inside the notebook, so no external download is required.

## 1. Import the libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

## 2. Create an intentionally imperfect dataset

Real datasets often contain multiple data-quality problems at the same time.  
The sample below includes:

- missing values;
- duplicate records;
- inconsistent category labels;
- prices stored as text;
- invalid negative quantities;
- inconsistent date formats;
- a suspiciously large order value.

In [ ]:
raw_data = {
    "order_id": [1001, 1002, 1003, 1004, 1005, 1005, 1006, 1007, 1008, 1009, 1010, 1011],
    "order_date": [
        "2026-01-03", "04/01/2026", "2026-01-05", "Jan 06 2026",
        "2026-01-07", "2026-01-07", None, "2026-01-09",
        "10-01-2026", "2026-01-11", "2026-01-12", "2026-01-13"
    ],
    "region": [
        "North", "north", " SOUTH ", "South", "East", "East",
        "West", None, "west", "North", "NORTH", "South"
    ],
    "product": [
        "Laptop", "Mouse", "Keyboard", "Monitor", "Laptop", "Laptop",
        "Mouse", "Keyboard", "Monitor", "Laptop", None, "Mouse"
    ],
    "unit_price": [
        "$1200", "$25", "$80", "$300", "$1,250", "$1,250",
        "$30", "$75", "$310", "$15000", "$1200", None
    ],
    "quantity": [1, 2, 1, 1, 1, 1, -3, 2, 1, 1, np.nan, 4],
    "customer_rating": [5, 4, np.nan, 4, 5, 5, 3, 6, 4, 2, 5, 4]
}

df = pd.DataFrame(raw_data)
df

## 3. Perform an initial inspection

Before changing data, inspect it. A useful first inspection normally includes:

- sample rows;
- number of rows and columns;
- column names;
- data types;
- missing-value counts;
- summary statistics;
- duplicate counts.

In [ ]:
print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isna().sum())

print("\nExact duplicate rows:", df.duplicated().sum())
print("Duplicated order IDs:", df["order_id"].duplicated().sum())

In [ ]:
df.describe(include="all").T

### Interpretation

Several issues are visible:

1. `order_date` is stored as text rather than as a datetime value.
2. `unit_price` is stored as text because it includes currency symbols and commas.
3. `region` contains inconsistent capitalization and extra spaces.
4. Some values are missing.
5. Order ID `1005` appears twice.
6. One quantity is negative.
7. One customer rating is outside the expected 1–5 scale.
8. The price `$15000` may be valid, but it should be reviewed as a potential outlier.

A value should not be removed merely because it looks unusual. Cleaning decisions require context.

## 4. Work on a copy

Keeping the original DataFrame unchanged makes the workflow safer and easier to audit.

In [ ]:
clean_df = df.copy()

## 5. Remove duplicate records

Two kinds of duplication may occur:

- **exact duplicates**, where all values are repeated;
- **business-key duplicates**, where an identifier such as `order_id` occurs more than once.

Here, the two rows with order ID `1005` are exact duplicates, so one can be removed.

In [ ]:
duplicate_rows = clean_df[clean_df.duplicated(keep=False)]
duplicate_rows

In [ ]:
clean_df = clean_df.drop_duplicates().reset_index(drop=True)
print("Shape after removing exact duplicates:", clean_df.shape)

## 6. Standardize categorical text

Text categories should be normalized before counting, grouping, or modeling.  
A common first step is to:

1. remove leading and trailing spaces;
2. use consistent capitalization;
3. inspect the remaining unique values.

In [ ]:
clean_df["region"] = clean_df["region"].str.strip().str.title()
clean_df["product"] = clean_df["product"].str.strip().str.title()

print("Regions:", clean_df["region"].unique())
print("Products:", clean_df["product"].unique())

## 7. Convert columns to suitable data types

### 7.1 Convert prices from text to numbers

Currency symbols and thousands separators must be removed before numeric conversion.
`errors="coerce"` converts invalid or missing text to `NaN`, allowing it to be inspected later.

In [ ]:
clean_df["unit_price"] = (
    clean_df["unit_price"]
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .pipe(pd.to_numeric, errors="coerce")
)

clean_df[["unit_price"]].head()

### 7.2 Convert dates

The source uses several date formats. Pandas can parse mixed formats, but ambiguous dates require care.  
This example assumes that slash- and dash-separated dates use day-first ordering.

In [ ]:
clean_df["order_date"] = pd.to_datetime(
    clean_df["order_date"],
    errors="coerce",
    dayfirst=True,
    format="mixed"
)

clean_df[["order_date"]].head()

In [ ]:
print(clean_df.dtypes)

## 8. Identify invalid values

Domain rules help distinguish missing values from invalid values.

For this example:

- quantity must be greater than zero;
- customer rating must be between 1 and 5;
- unit price must be greater than zero.

In [ ]:
invalid_quantity = clean_df["quantity"].le(0)
invalid_rating = ~clean_df["customer_rating"].between(1, 5, inclusive="both")
invalid_price = clean_df["unit_price"].le(0)

print("Invalid quantities:")
display(clean_df.loc[invalid_quantity])

print("Invalid ratings:")
display(clean_df.loc[invalid_rating & clean_df["customer_rating"].notna()])

print("Invalid prices:")
display(clean_df.loc[invalid_price])

Invalid values should usually be investigated. In this teaching example, they are converted to missing values so that an explicit missing-value strategy can be applied.

In [ ]:
clean_df.loc[invalid_quantity, "quantity"] = np.nan
clean_df.loc[
    invalid_rating & clean_df["customer_rating"].notna(),
    "customer_rating"
] = np.nan
clean_df.loc[invalid_price, "unit_price"] = np.nan

## 9. Analyze missing values

Counts show the number of missing values. Percentages make comparisons easier when datasets are large.

In [ ]:
missing_summary = pd.DataFrame({
    "missing_count": clean_df.isna().sum(),
    "missing_percentage": clean_df.isna().mean().mul(100).round(2)
}).sort_values("missing_percentage", ascending=False)

missing_summary

### Missing-value strategies

The appropriate treatment depends on the variable, the amount of missingness, and the analytical purpose.

Common strategies include:

- **remove rows** when only a small number of non-critical observations are incomplete;
- **remove columns** when a feature is mostly missing and not essential;
- **mean imputation** for approximately symmetric numerical variables;
- **median imputation** for skewed numerical variables or variables affected by outliers;
- **mode imputation** for categorical variables;
- **group-based imputation** when a value depends on another category;
- **explicit “Unknown” category** when missingness itself may be informative.

Imputation changes the data and can reduce variability. It should always be documented.

## 10. Apply transparent missing-value treatments

For this example:

- missing `region` and `product` values become `"Unknown"`;
- missing `quantity` is replaced by the median quantity;
- missing `customer_rating` is replaced by the median rating;
- missing `unit_price` is replaced by the median price for the same product when available, then by the overall median;
- a row with a missing order date is retained and flagged rather than silently removed.

In [ ]:
clean_df["region"] = clean_df["region"].fillna("Unknown")
clean_df["product"] = clean_df["product"].fillna("Unknown")

clean_df["quantity"] = clean_df["quantity"].fillna(clean_df["quantity"].median())
clean_df["customer_rating"] = clean_df["customer_rating"].fillna(
    clean_df["customer_rating"].median()
)

product_price_median = clean_df.groupby("product")["unit_price"].transform("median")
clean_df["unit_price"] = clean_df["unit_price"].fillna(product_price_median)
clean_df["unit_price"] = clean_df["unit_price"].fillna(clean_df["unit_price"].median())

clean_df["order_date_missing"] = clean_df["order_date"].isna()

Because quantities represent item counts, convert the imputed quantity back to an integer type after confirming that the median is a whole number.

In [ ]:
clean_df["quantity"] = clean_df["quantity"].round().astype("Int64")

## 11. Detect potential outliers using the IQR rule

The interquartile range is:

\[
IQR = Q_3 - Q_1
\]

A common screening rule flags values below:

\[
Q_1 - 1.5(IQR)
\]

or above:

\[
Q_3 + 1.5(IQR)
\]

This rule identifies observations for review; it does not prove that they are errors.

In [ ]:
q1 = clean_df["unit_price"].quantile(0.25)
q3 = clean_df["unit_price"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

price_outliers = clean_df[
    (clean_df["unit_price"] < lower_bound) |
    (clean_df["unit_price"] > upper_bound)
]

print(f"Lower bound: {lower_bound:.2f}")
print(f"Upper bound: {upper_bound:.2f}")
price_outliers

In [ ]:
clean_df["unit_price"].plot(
    kind="box",
    title="Unit-price distribution and potential outliers",
    ylabel="Unit price"
)
plt.show()

The `$15000` value is retained because the dataset alone does not establish that it is incorrect.  
A real analyst should compare it with the source system, product catalogue, invoice, or domain expert.

## 12. Create useful derived variables

Once the principal quality issues are handled, derived features can be calculated safely.

In [ ]:
clean_df["order_value"] = clean_df["unit_price"] * clean_df["quantity"]

clean_df[[
    "order_id", "product", "unit_price", "quantity", "order_value"
]].head()

## 13. Validate the cleaned dataset

A cleaning workflow should end with explicit validation rather than visual inspection alone.

In [ ]:
assert clean_df["order_id"].is_unique, "Order IDs must be unique."
assert clean_df["quantity"].gt(0).all(), "Quantities must be positive."
assert clean_df["customer_rating"].between(1, 5).all(), "Ratings must be between 1 and 5."
assert clean_df["unit_price"].gt(0).all(), "Prices must be positive."
assert clean_df[["region", "product", "unit_price", "quantity", "customer_rating"]].notna().all().all()

print("All validation checks passed.")

In [ ]:
print("Cleaned shape:", clean_df.shape)
print("\nRemaining missing values:")
print(clean_df.isna().sum())

clean_df

## 14. Summarize the cleaning decisions

In [ ]:
cleaning_log = pd.DataFrame({
    "issue": [
        "Exact duplicate",
        "Inconsistent region labels",
        "Currency-formatted prices",
        "Mixed date formats",
        "Invalid negative quantity",
        "Invalid rating above 5",
        "Missing categorical values",
        "Missing numerical values",
        "Potential price outlier"
    ],
    "action": [
        "Removed one exact duplicate row",
        "Trimmed spaces and standardized capitalization",
        "Removed symbols and converted to numeric",
        "Parsed using mixed formats with day-first assumption",
        "Converted to missing, then imputed with median",
        "Converted to missing, then imputed with median",
        "Replaced with 'Unknown'",
        "Applied documented median-based imputations",
        "Flagged for review and retained"
    ]
})

cleaning_log

## 15. Save the cleaned data

In a real project, avoid overwriting the original raw file. Save cleaned data separately and preserve the cleaning code.

In [ ]:
output_path = "cleaned_retail_orders.csv"
clean_df.to_csv(output_path, index=False)
print(f"Saved cleaned dataset to: {output_path}")

## 16. Guided exercises

1. Replace the median quantity imputation with product-specific median imputation.
2. Create a table comparing missing values before and after cleaning.
3. Add a rule that checks whether every `order_id` is an integer.
4. Investigate how removing the `$15000` observation changes the mean and median unit price.
5. Create a bar chart of total order value by region.
6. Add a `data_quality_status` column that marks rows requiring manual review.
7. Explain why a missing order date was flagged rather than imputed.

## 17. Key conclusions

- Data cleaning begins with inspection, not immediate modification.
- Missing and invalid values are related but not identical.
- Data types must reflect the meaning of each variable.
- Category normalization is essential before aggregation.
- Duplicate removal should use a justified business key or exact-row rule.
- Outliers are observations for investigation, not automatic deletion targets.
- Every cleaning decision should be reproducible, validated, and documented.